# Step Overrides

## What you'll learn

- Override operation parameters, names, and execution config per step
- Control failure handling with `FailurePolicy`
- Understand which defaults each step override replaces

**Prerequisites:** [First Pipeline](../01-getting-started/01-first-pipeline.ipynb),
[Batching and Performance](../04-batching/01-batching-and-performance.ipynb).
**Estimated time:** 15 minutes
**GPU required:** No.

---

Pass overrides to `pipeline.run()` when a step needs different settings from
the operation or pipeline defaults.

In [ ]:
from __future__ import annotations

from artisan.operations.examples import (
    DataGenerator,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import PipelineManager
from artisan.orchestration.runners import LocalRunner
from artisan.schemas import FailurePolicy, GroupByStrategy
from artisan.utils import tutorial_setup
from artisan.visualization import build_macro_graph, build_micro_graph, inspect_pipeline

In [ ]:
from functools import partial
from tempfile import TemporaryDirectory

# Keep this tutorial's stores and source files in one owned temporary workspace.
workspace = TemporaryDirectory(prefix="artisan-tutorial-")
tutorial_setup = partial(tutorial_setup, base_dir=workspace.name)

In [ ]:
env = tutorial_setup("step_overrides")

## Choose an override

Use step overrides when the operation’s reusable defaults do not fit a particular
invocation. Start with the examples below. For other settings, see
[Configure Execution](../../how-to-guides/configuring-execution.md) and the current
`PipelineManager.run` docstring linked from the [Python API guide](../../reference/python-api.md).

## `params` — operation parameters

An operation’s parameter model supplies defaults. Pass `params` to change
values for this step; omit it to use the operation defaults. Parameters declared
as required still need values. Here we change the generator’s count and seed.

In [ ]:
pipeline = PipelineManager.create(
    name="params_demo",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)

# Same operation, different params → different outputs
pipeline.run(
    definition=DataGenerator,
    name="small",
    params={"count": 3, "seed": 42},
    step_runner=LocalRunner(),
)
pipeline.run(
    definition=DataGenerator,
    name="large",
    params={"count": 10, "seed": 99},
    step_runner=LocalRunner(),
)

pipeline.finalize()
inspect_pipeline(env.delta_root)

Both steps use `DataGenerator` but produce different outputs: step 0 generates
3 datasets with seed 42, step 1 generates 10 with seed 99. The pipeline
overview shows the different artifact counts.


## `name` — custom step name

By default, each step is named after the operation (e.g., `data_generator`).
Use `name` to give steps descriptive labels, especially when the same
operation appears multiple times.


In [ ]:
env_name = tutorial_setup("step_names")

pipeline = PipelineManager.create(
    name="name_demo",
    delta_root=env_name.delta_root,
    staging_root=env_name.staging_root,
    working_root=env_name.working_root,
)
output = pipeline.output

pipeline.run(
    definition=DataGenerator,
    name="initial_candidates",
    params={"count": 5, "seed": 42},
    step_runner=LocalRunner(),
)
pipeline.run(
    definition=DataTransformer,
    name="normalize",
    inputs={"dataset": output("initial_candidates", "datasets")},
    step_runner=LocalRunner(),
)
pipeline.run(
    definition=DataTransformer,
    name="augment",
    inputs={"dataset": output("normalize", "dataset")},
    params={"seed": 100},
    step_runner=LocalRunner(),
)

pipeline.finalize()
inspect_pipeline(env_name.delta_root)

The overview shows `initial_candidates`, `normalize`, and `augment`. These
names also appear in provenance graphs.

In [ ]:
build_macro_graph(env_name.delta_root)

## `batch_strategy` — batching configuration

The `batch_strategy` dict accepts any `BatchStrategy` field. See
[Batching and Performance](../04-batching/01-batching-and-performance.ipynb) for a batching example.


In [ ]:
env_exec = tutorial_setup("step_execution")

pipeline = PipelineManager.create(
    name="execution_demo",
    delta_root=env_exec.delta_root,
    staging_root=env_exec.staging_root,
    working_root=env_exec.working_root,
)
output = pipeline.output

pipeline.run(
    definition=DataGenerator,
    name="generate",
    params={"count": 12, "seed": 42},
    step_runner=LocalRunner(),
)

# Override batching: 4 artifacts per unit, max 2 concurrent workers
pipeline.run(
    definition=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("generate", "datasets")},
    batch_strategy={"artifacts_per_unit": 4, "max_workers": 2},
    step_runner=LocalRunner(),
)

pipeline.finalize()
inspect_pipeline(env_exec.delta_root)

Step 1 groups 12 artifacts into 3 execution units (4 per unit) and runs at
most 2 concurrently. Without the override, MetricCalculator's default
`artifacts_per_unit` of 10,000 would put all 12 artifacts into a single unit.


## Override the worker runner

Supply a live runner instance for the selected step. For example, a pipeline
with a configured remote default can run one step locally:

```python
pipeline.run(
    MetricCalculator,
    inputs={"dataset": output("generate", "datasets")},
    step_runner=LocalRunner(),
)
```

Core provides `LocalRunner`; matching optional packages provide Modal or Slurm
runners. Prepare required software explicitly before submission.

## Resources belong to the worker

Use `runner_resources` for the worker's CPU, memory, GPU, and time requirements.
Providers validate and translate supported requests. The local runner observes
actual local hardware; a resource request cannot create missing devices.

Command environments are declared on operations and activated inside the worker.
Changing worker hardware or software may require preparation again. See
[Configure Execution](../../how-to-guides/configuring-execution.md).

## `failure_policy` — handling failures

`CONTINUE` keeps accepted successful outputs when some work fails. `FAIL_FAST`
makes an attempt with an observed failure terminally failed and exposes no
accepted outputs. Finished execution evidence remains available for diagnosis.
Both policies return a `StepResult`; invalid configuration can still raise an
exception before execution.

The next example overrides the pipeline policy for its transform step.

In [ ]:
env_fp = tutorial_setup("step_failure_policy")

pipeline = PipelineManager.create(
    name="failure_demo",
    delta_root=env_fp.delta_root,
    staging_root=env_fp.staging_root,
    working_root=env_fp.working_root,
    failure_policy=FailurePolicy.CONTINUE,  # Pipeline default
)
output = pipeline.output

pipeline.run(
    definition=DataGenerator,
    name="generate",
    params={"count": 5, "seed": 42},
    step_runner=LocalRunner(),
)

# This step must succeed completely — fail fast on any error
pipeline.run(
    definition=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    failure_policy=FailurePolicy.FAIL_FAST,
    step_runner=LocalRunner(),
)

# This step tolerates partial failures — continue processing
pipeline.run(
    definition=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
    failure_policy=FailurePolicy.CONTINUE,
    step_runner=LocalRunner(),
)

pipeline.finalize()
inspect_pipeline(env_fp.delta_root)

All steps succeed in this example. If the transform failed, its `FAIL_FAST`
result would have `status=FAILED` and no accepted output roles. The metrics
step’s `CONTINUE` policy allows partial successful output when applicable.
[Error Handling in Practice](02-error-visibility.ipynb) demonstrates actual
failures and inspects their persisted evidence.

## `group_by` — pairing strategy for multi-input steps

The next operation has two inputs. Its step overrides `group_by` with
`CROSS_PRODUCT`, so two left inputs and three right inputs produce six pairs.
The operation declares no pairing default of its own.

[Multi-Input Operations](../02-pipeline-design/04-multi-input-operations.ipynb)
and [Name-Based Pairing](../02-pipeline-design/05-name-based-pairing.ipynb)
show pairing by ancestry and filename when only matching inputs should join.
Changing the strategy changes the step’s cache identity as well.

In [ ]:
from fsspec.implementations.local import LocalFileSystem

from artisan.operations.examples import CsvPair
from artisan.storage import ArtifactStore

for artifacts_per_unit in (1, 3):
    env_groupby = tutorial_setup(f"step_group_by_{artifacts_per_unit}")

    pipeline = PipelineManager.create(
        name="group_by_demo",
        delta_root=env_groupby.delta_root,
        staging_root=env_groupby.staging_root,
        working_root=env_groupby.working_root,
    )
    output = pipeline.output

    # Two independent DataGenerator runs — no shared ancestry between them,
    # so LINEAGE pairing would match nothing.
    pipeline.run(
        definition=DataGenerator,
        name="gen_left",
        params={"count": 2, "seed": 42},
        step_runner=LocalRunner(),
    )
    pipeline.run(
        definition=DataGenerator,
        name="gen_right",
        params={"count": 3, "seed": 100},
        step_runner=LocalRunner(),
    )

    # 2 left x 3 right = 6 paired outputs via per-step CROSS_PRODUCT.
    pipeline.run(
        definition=CsvPair,
        name="pair_all",
        inputs={
            "left": output("gen_left", "datasets"),
            "right": output("gen_right", "datasets"),
        },
        group_by=GroupByStrategy.CROSS_PRODUCT,
        batch_strategy={"artifacts_per_unit": artifacts_per_unit},
        step_runner=LocalRunner(),
    )

    pipeline.finalize()
    store = ArtifactStore(env_groupby.delta_root, fs=LocalFileSystem())
    by_step = []
    for step_number in (0, 1, 2):
        ids = store.provenance.load_artifact_ids_by_type(
            "csv", step_numbers=[step_number]
        )
        artifacts = store.get_artifacts_by_type(list(ids), "csv")
        by_step.append({a.original_name: a.content for a in artifacts.values()})
    left, right, actual = by_step
    expected = {
        left_bytes + b"\n---\n" + right_bytes
        for left_bytes in left.values()
        for right_bytes in right.values()
    }
    assert len(actual) == 6
    assert set(actual.values()) == expected
    print(f"Verified all six pairs with {artifacts_per_unit} pair(s) per unit")

inspect_pipeline(env_groupby.delta_root)

In [ ]:
build_macro_graph(env_groupby.delta_root)

In [ ]:
build_micro_graph(env_groupby.delta_root)

Each run produced the same six exact byte pairs. `PerArtifact` marks the
paths that must be sliced for each execute call; plain lists are shared values.
That declaration keeps the pairing correct with one or three pairs per unit.
The graphs show the second run.

## Run a complete worker remotely

Pass a configured provider through `step_runner`. It runs the complete
preprocess, execute, and postprocess lifecycle. A composite can use several tool
environments inside that worker without launching remote workers for its children.
There is no separate per-operation execute-only endpoint declaration.

[Running on Modal](../07-compute-backends/04-modal-execution.ipynb) shows explicit
preparation, resource binding, shared storage, and accepted-output checks.

## Configure a command environment

An `environment` override chooses how a command runs, such as a configured
Docker image. A `tool` override changes fields on an operation’s existing tool
declaration, such as its executable path. These require the corresponding local
software or image. Follow [Configure Execution](../../how-to-guides/configuring-execution.md)
for a complete recipe.

## Where defaults come from

Explicit step values take precedence. Their default source differs:

- `step_runner`, `failure_policy`, and `cache_policy` inherit from the pipeline.
- Batching, resources, pairing, environment, tool, and compute settings start
  from the operation declaration.
- `params` uses the operation’s parameter defaults unless you supply values.

For example, a step can require a fully successful cached result even when the
pipeline permits partial results:

```python
from artisan.schemas import CachePolicy

pipeline.run(DataGenerator, cache_policy=CachePolicy.ALL_SUCCEEDED)
```

[Resume and Caching](../03-caching/01-resume-and-caching.ipynb) demonstrates both
the pipeline default and a step override.

## Next steps

- [Batching and Performance](../04-batching/01-batching-and-performance.ipynb) — Compare batch sizes and worker packing
- [Configuring Execution](../../how-to-guides/configuring-execution.md) — Execution configuration recipes
- [Multi-Input Operations](../02-pipeline-design/04-multi-input-operations.ipynb) — LINEAGE pairing in depth
- [Name-Based Pairing](../02-pipeline-design/05-name-based-pairing.ipynb) — NAME pairing in depth
- [Error Handling in Practice](02-error-visibility.ipynb) — Runtime failures, failure logs, and FailurePolicy
